# Function Calling Fine-Tuning for Qwen3-1.7B

Fine-tune Qwen3-1.7B for reliable tool calling using the Strands SDK format, optimized for edge deployment with llama.cpp.

## Pipeline Overview

1. **Data Generation**: Synthetic training data using foundation models
2. **LoRA Training**: Efficient fine-tuning on consumer GPUs (8GB VRAM)
3. **Quantization**: Q4_K_M compression for edge deployment (3.5GB → 1.1GB)
4. **Validation**: Tool calling accuracy and performance metrics

## Environment Setup

In [ ]:
# AWS Configuration
import os

os.environ['AWS_BEARER_TOKEN_BEDROCK'] = 'YOUR_BEARER_TOKEN_HERE'
os.environ['AWS_REGION'] = 'us-east-1'

MODEL_CONFIG = {
    'model_id': 'us.anthropic.claude-3-5-sonnet-20241022-v2:0',
    'max_tokens': 1000,
    'temperature': 0.7
}

# Install dependencies
!pip install -q torch transformers>=4.52.3 accelerate datasets trl peft boto3 httpx matplotlib
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" || pip install -q unsloth

In [ ]:
import sys
import json
import torch
from pathlib import Path
from typing import Dict, List, Any

sys.path.append('./utils')

from data_generator import DataGenerator, ToolRegistry
from trainer import ModelTrainer, TrainingConfig  
from quantizer import ModelQuantizer, QuantizationConfig
from evaluator import ModelEvaluator, EvaluationMetrics

# System info
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB VRAM)")

# Create directories
for directory in ["./data", "./models", "./outputs"]:
    Path(directory).mkdir(exist_ok=True)

## Stage 1: Data Generation

In [ ]:
# Initialize components
registry = ToolRegistry()
generator = DataGenerator(registry)

# Display available tools
print(f"Tools configured: {len(registry.tools)}")
for name, spec in registry.tools.items():
    params = list(spec.parameters['properties'].keys())
    print(f"  - {name}({', '.join(params)})")

In [ ]:
# Generate datasets
train_size = 200  # Increase to 1000+ for production
test_size = 50    # Increase to 200+ for production

generator.generate_dataset(num_examples=train_size, output_path="./data/train.jsonl")
generator.generate_dataset(num_examples=test_size, output_path="./data/test.jsonl")

print(f"Generated {train_size + test_size} examples")

In [ ]:
# Analyze data quality
import matplotlib.pyplot as plt
from collections import Counter

with open("./data/train.jsonl", "r") as f:
    train_data = [json.loads(line) for line in f]

# Tool distribution
tool_counts = Counter()
for example in train_data:
    for tool in example.get("tools", []):
        tool_counts[tool["name"]] += 1

# Visualize
plt.figure(figsize=(10, 4))
tools, counts = zip(*tool_counts.most_common())
plt.bar(range(len(tools)), counts)
plt.xticks(range(len(tools)), tools, rotation=45, ha='right')
plt.title('Tool Distribution in Training Data')
plt.tight_layout()
plt.show()

## Stage 2: LoRA Fine-Tuning

In [ ]:
# Training configuration
config = TrainingConfig(
    model_name="Qwen/Qwen3-1.7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    num_epochs=2,  # Increased from 1
    batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    output_dir="./models/fine-tuned",
    save_strategy="steps",
    save_steps=50,
    use_flash_attention=True,
    gradient_checkpointing=True
)

config.save("./models/training_config.json")

In [ ]:
# Setup and train
trainer = ModelTrainer(config)
trainer.setup_model()

print(f"Trainable parameters: {trainer.param_ratio:.2f}%")

In [ ]:
# Prepare datasets and train
train_dataset, eval_dataset = trainer.prepare_dataset("./data/train.jsonl")
training_history = trainer.train(train_dataset, eval_dataset)

In [ ]:
# Plot training metrics
train_loss = [h['loss'] for h in training_history if 'loss' in h]
eval_loss = [h['eval_loss'] for h in training_history if 'eval_loss' in h]

if train_loss:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_loss)
    plt.title('Training Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    
    if eval_loss:
        plt.subplot(1, 2, 2)
        plt.plot(eval_loss)
        plt.title('Evaluation Loss')
        plt.xlabel('Evaluation Steps')
        plt.ylabel('Loss')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Merge LoRA weights
merged_model = trainer.merge_and_save("./models/merged")
print("Model merged and saved")

## Stage 3: Quantization

In [ ]:
# Configure quantization
quant_config = QuantizationConfig(
    quantization_method="q4_k_m",
    use_mmap=True,
    include_mmproj=True
)

quantizer = ModelQuantizer(quant_config)

if not quantizer.check_dependencies():
    print("Install llama.cpp: git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make")

In [ ]:
# Run quantization
results = quantizer.full_pipeline(
    model_path="./models/merged",
    output_dir="./outputs/gguf",
    model_name="qwen3-1.7b-finetuned"
)

# Display results
for key, path in results.items():
    if Path(path).exists():
        size_gb = Path(path).stat().st_size / (1024**3)
        print(f"{key}: {Path(path).name} ({size_gb:.2f} GB)")

## Stage 4: Evaluation

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluator(
    base_url="http://localhost:8080",
    test_data_path="./data/test.jsonl"
)

print(f"Server command: llama-server -m {results.get('quantized', 'model.gguf')} --host 0.0.0.0 --port 8080 -c 2048 -ngl 35")

In [ ]:
# Run evaluation (requires server running)
if evaluator.client:
    metrics = evaluator.run_full_evaluation()
    metrics.save("./outputs/evaluation_metrics.json")
    
    print(f"Tool Accuracy: {metrics.tool_accuracy:.1%}")
    print(f"Format Validity: {metrics.format_validity:.1%}")
    print(f"Avg Latency: {metrics.avg_latency_ms:.0f}ms")
    print(f"Throughput: {metrics.tokens_per_second:.1f} tok/s")

## Deployment

### Start Server
```bash
llama-server -m outputs/gguf/qwen3-1.7b-finetuned-q4_k_m.gguf \
  --host 0.0.0.0 --port 8080 -c 2048 -ngl 35 --jinja
```

### Python Integration
```python
from strands.models.llamacpp import LlamaCppModel

model = LlamaCppModel(
    base_url="http://localhost:8080",
    params={'temperature': 0.7, 'max_tokens': 1024}
)
```

### Performance Targets
- Tool Selection Accuracy: >95%
- Format Validity: >99%
- Inference Speed: 20-35 tok/s
- Memory Usage: <2GB
- Model Size: ~1.1GB (Q4_K_M)